In [17]:
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')


In [18]:
# --- COMPILA QUESTI PARAMETRI IN BASE AL TUO TRAINING ---
NUM_ASSETS = 362  
HIDDEN_DIMS = [2048, 1024, 512]
LATENT_DIM = 20
FILE_NAME = "data_00_20"
WINDOW_LENGTH = 724
STRIDE = 1
DATASET_NAME = f"{FILE_NAME}_w{WINDOW_LENGTH}_s{STRIDE}"
DATASET_MODEL_NAME = f"{FILE_NAME}_w{WINDOW_LENGTH}_s10"
INPUT_DIM = NUM_ASSETS * (NUM_ASSETS + 1) // 2
RUN_NAME = "VAE_20dim_cholesky_06_loss" 
N_MATRICES = 100

In [19]:
class VAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], latent_dim: int):
        super().__init__()

        # Encoder
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers.append(nn.Linear(prev, h))
            enc_layers.append(nn.LeakyReLU(0.01))
            prev = h
        self.encoder = nn.Sequential(*enc_layers)

        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        # Decoder
        dec_layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers.append(nn.Linear(prev, h))
            dec_layers.append(nn.LeakyReLU(0.01))
            prev = h
        dec_layers.append(nn.Linear(prev, input_dim))
        #dec_layers.append(nn.Tanh())
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x: torch.Tensor):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        sample = mu + (eps * std)
        return sample

    def decode(self, z: torch.Tensor):
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        if self.training:
            # Durante l'addestramento (model.train()), usa il campionamento stocastico
            z = self.reparameterize(mu, logvar)
        else:
            # Durante la validazione/test (model.eval()), usa l'aspettativa esatta
            z = mu
        x_hat = self.decode(z)
        return x_hat, mu, logvar

In [20]:
# Setup del device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resume_source_path = Path(rf"..\models\{DATASET_MODEL_NAME}\VAE\{RUN_NAME}\best_model.pt")

# --- LOAD E CONTROLLO DEL CHECKPOINT ---
checkpoint = torch.load(resume_source_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint

# Inizializziamo il modello (funge anche da probe_model per i check)
model = VAE(
    input_dim=INPUT_DIM,
    latent_dim=LATENT_DIM,
    hidden_dims=HIDDEN_DIMS,
)

current_shapes = {k: tuple(v.shape) for k, v in model.state_dict().items()}
loaded_shapes = {k: tuple(v.shape) for k, v in state_dict.items()}

missing_keys = [k for k in current_shapes.keys() if k not in loaded_shapes]
unexpected_keys = [k for k in loaded_shapes.keys() if k not in current_shapes]
shape_mismatches = [
    (k, loaded_shapes[k], current_shapes[k])
    for k in current_shapes.keys()
    if k in loaded_shapes and loaded_shapes[k] != current_shapes[k]
]

if missing_keys or shape_mismatches or unexpected_keys:
    suggested_hidden = []
    # Cerchiamo tutti i pesi dei layer lineari nell'encoder
    for key, weight in state_dict.items():
        if key.startswith('encoder.') and key.endswith('.weight'):
            # Escludiamo il primo strato (input_dim -> hidden[0])
            if weight.shape[1] == INPUT_DIM:
                continue
            suggested_hidden.append(int(weight.shape[0]))

    # L'ultimo elemento trovato sara il latent_dim
    if suggested_hidden:
        ckpt_latent_dim = suggested_hidden[-1]
        ckpt_hidden_dims = suggested_hidden[:-1]
        hint = (
            f'Checkpoint expects latent_dim={ckpt_latent_dim}, '
            f'hidden_dims={ckpt_hidden_dims}. '
            'Set these values to match before resuming, or start a new run without resume.'
        )
    else:
        hint = 'Checkpoint architecture could not be inferred. Use a checkpoint saved with the same model architecture.'

    details = []
    if missing_keys:
        details.append(f'Missing keys (first 5): {missing_keys[:5]}')
    if unexpected_keys:
        details.append(f'Unexpected keys (first 5): {unexpected_keys[:5]}')
    if shape_mismatches:
        k, loaded_shape, current_shape = shape_mismatches[0]
        details.append(f'Shape mismatch example: {k} loaded={loaded_shape}, current={current_shape}')

    raise RuntimeError(
        'Incompatible checkpoint for current VAE architecture. ' + hint + (' | ' + ' ; '.join(details) if details else '')
    )
else:
    # Se i controlli sulle shape passano, applichiamo i pesi
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print("Controllo di integrità superato: Pesi caricati correttamente.")
    print(f"Modello impostato in modalità eval() sul device: {device}")

Controllo di integrità superato: Pesi caricati correttamente.
Modello impostato in modalità eval() sul device: cuda


In [21]:
print(model)

VAE(
  (encoder): Sequential(
    (0): Linear(in_features=65703, out_features=2048, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=2048, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
  )
  (fc_mu): Linear(in_features=512, out_features=20, bias=True)
  (fc_logvar): Linear(in_features=512, out_features=20, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=20, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=2048, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=2048, out_features=65703, bias=True)
  )
)


In [22]:
def reconstruct_correlation_matrices(recon_flat, n_assets):
    """Ricostruisce le matrici di correlazione garantendo PSD e normalizzando la diagonale."""
    if hasattr(recon_flat, 'detach'):
        recon_flat = recon_flat.detach().cpu().numpy()
        
    n_samples = recon_flat.shape[0]
    
    # Inizializza il batch di matrici quadrate L (Cholesky factor) riempite di zeri
    recon_L = np.zeros((n_samples, n_assets, n_assets), dtype=np.float32)

    # Estrai indici del triangolo inferiore INCLUSA la diagonale (k=0)
    tril_idx = np.tril_indices(n_assets, k=0)

    # Inserisci i valori predetti per formare le matrici triangolari inferiori
    recon_L[:, tril_idx[0], tril_idx[1]] = recon_flat

    # RICOSTRUZIONE E NORMALIZZAZIONE DELLA CORRELAZIONE (Garantisce PSD)
    # Calcolo di C = L @ L.T 
    recon_PSD = recon_L @ np.transpose(recon_L, axes=(0, 2, 1))

    # Estraiamo la diagonale di ogni matrice (le varianze)
    d = np.diagonal(recon_PSD, axis1=1, axis2=2)
    
    # Calcoliamo 1 / sqrt(d) gestendo eventuali divisioni per zero
    d_inv_sqrt = 1.0 / np.sqrt(np.maximum(d, 1e-12))
    
    # Creiamo matrici diagonali per la normalizzazione
    D_inv_sqrt = np.zeros_like(recon_PSD)
    diag_idx = np.arange(n_assets)
    D_inv_sqrt[:, diag_idx, diag_idx] = d_inv_sqrt
    
    # Normalizzazione: Corr = D_inv_sqrt @ PSD @ D_inv_sqrt
    recon_corr = D_inv_sqrt @ recon_PSD @ D_inv_sqrt
    
    # Pulizia floating point finale e ripristino esatto della diagonale
    recon_corr = np.clip(recon_corr, -1.0, 1.0)
    recon_corr[:, diag_idx, diag_idx] = 1.0
    
    return recon_corr

In [23]:
# =====================================================================
# SEZIONE BLOCK BOOTSTRAPPING STORICO NELLO SPAZIO LATENTE
# =====================================================================

print("\n--- Inizio Block Bootstrapping Storico (Orizzonte: 1 Mese / 21 gg) ---")

# 1. Caricamento del dataset reale
dataset_path = Path(rf"..\data\processed\{FILE_NAME}\dataset\{DATASET_NAME}\all.pt")
print(f"Caricamento dataset da: {dataset_path}")

try:
    payload = torch.load(dataset_path, map_location=device, weights_only=False)
except TypeError:
    payload = torch.load(dataset_path, map_location=device)

# Carichiamo direttamente dal tensore di Cholesky 3D
historical_matrices = payload['corr_tensor'].to(device).float()
historical_data = payload['chol_tensor'].to(device).float()

# Estrazione del triangolo inferiore inclusa la diagonale
if historical_data.ndim == 3:
    N_MATRICES, N_ASSETS, _ = historical_data.shape
    tril_idx = torch.tril_indices(row=N_ASSETS, col=N_ASSETS, offset=0)
    historical_data = historical_data[:, tril_idx[0], tril_idx[1]]

print(f"Shape dei dati storici flattenati per l'Encoder: {historical_data.shape}")

# --- CARICAMENTO STATISTICHE E NORMALIZZAZIONE ---
mean_path = Path(rf"..\models\{DATASET_MODEL_NAME}\VAE\{RUN_NAME}\train_mean.pt")
std_path = Path(rf"..\models\{DATASET_MODEL_NAME}\VAE\{RUN_NAME}\train_std.pt")

train_mean = torch.load(mean_path, map_location=device, weights_only=True)
train_std = torch.load(std_path, map_location=device, weights_only=True)

# Applichiamo la Z-score normalization 
historical_data_norm = (historical_data - train_mean) / train_std
print("Dati storici normalizzati con successo.")

N_SCENARIOS = 100 
BLOCK_SIZE = 7    # Lunghezza del singolo blocco di incrementi consecutivi (es. 1 settimana)
N_BLOCKS = 3      # Quanti blocchi concatenare (7 gg * 3 blocchi = 21 giorni lavorativi)
TARGET_DAYS = BLOCK_SIZE * N_BLOCKS

with torch.no_grad():
    # 2. Codifica dell'intera storia (otteniamo la traiettoria latente continua)
    z_history, _ = model.encode(historical_data_norm)
    
    # 3. Calcolo degli incrementi GIORNALIERI (poiché ora stride=1)
    daily_deltas = z_history[1:] - z_history[:-1]
    n_days = daily_deltas.shape[0]
    
    # Calcoliamo quanti blocchi di lunghezza BLOCK_SIZE possiamo estrarre
    n_possible_blocks = n_days - BLOCK_SIZE + 1
    
    if n_possible_blocks <= 0:
        raise ValueError("Non ci sono abbastanza dati storici per creare i blocchi richiesti.")
        
    # 4. Stato attuale (il punto di partenza per le simulazioni)
    z_oggi = z_history[-1]
    
    # Inizializziamo il tensore che conterrà i vettori latenti futuri (scenari)
    z_futuro = torch.zeros((N_SCENARIOS, z_oggi.shape[0]), device=device)
    sampled_cumulative_deltas = torch.zeros_like(z_futuro)
    
    # 5. ESECUZIONE DEL BLOCK BOOTSTRAP
    for i in range(N_SCENARIOS):
        scenario_cumulative_delta = torch.zeros_like(z_oggi)
        
        # Estraiamo casualmente N_BLOCKS indici di partenza per i nostri blocchi
        random_starts = torch.randint(0, n_possible_blocks, (N_BLOCKS,), device=device)
        
        for start_idx in random_starts:
            # Estraiamo il blocco (sequenza di incrementi consecutivi)
            block = daily_deltas[start_idx : start_idx + BLOCK_SIZE]
            
            # Accumuliamo/Sommiamo l'intera sequenza di incrementi del blocco
            scenario_cumulative_delta += torch.sum(block, dim=0)
            
        # Salviamo gli incrementi totali e calcoliamo il punto d'arrivo z_futuro
        sampled_cumulative_deltas[i] = scenario_cumulative_delta
        z_futuro[i] = z_oggi + scenario_cumulative_delta
    
    # 6. Decodifica degli scenari simulati a 21 giorni
    generated_flat_matrices_norm = model.decode(z_futuro)

# --- DENORMALIZZAZIONE DELL'OUTPUT ---
generated_flat_matrices = (generated_flat_matrices_norm * train_std) + train_mean
print(f"Output del decoder denormalizzato. Scenari generati a {TARGET_DAYS} giorni.")
# ---------------------------------------------

# 7. Ricostruzione in matrici quadrate simmetriche
final_correlation_matrices = reconstruct_correlation_matrices(generated_flat_matrices, NUM_ASSETS)
print(f"Shape matrici di correlazione previste finali: {final_correlation_matrices.shape}")

# Salvataggio
output_save_path = Path(rf"..\generated_matrices\{DATASET_NAME}\{RUN_NAME}\forecast_scenarios_{N_SCENARIOS}\forecast_scenarios_{N_SCENARIOS}.pt")
output_save_path.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    'forecast_matrices': final_correlation_matrices,
    'z_oggi': z_oggi.cpu(),
    'matrice_oggi': historical_matrices[-1].cpu(),
    'sampled_cumulative_deltas': sampled_cumulative_deltas.cpu(), 
    'z_history': z_history.cpu()
}, output_save_path)

print(f"Scenari salvati con successo in: {output_save_path}")


--- Inizio Block Bootstrapping Storico (Orizzonte: 1 Mese / 21 gg) ---
Caricamento dataset da: ..\data\processed\data_00_20\dataset\data_00_20_w724_s1\all.pt
Shape dei dati storici flattenati per l'Encoder: torch.Size([4124, 65703])
Dati storici normalizzati con successo.
Output del decoder denormalizzato. Scenari generati a 21 giorni.
Shape matrici di correlazione previste finali: (100, 362, 362)
Scenari salvati con successo in: ..\generated_matrices\data_00_20_w724_s1\VAE_20dim_cholesky_06_loss\forecast_scenarios_100\forecast_scenarios_100.pt
